Docs:<br>
- [ReportLab Docs](https://docs.reportlab.com/reportlab/userguide/ch1_intro/)
- [StreamLit Gallery - voor ophalen van data](https://streamlit.io/gallery)
- [FPDF2 docs](https://py-pdf.github.io/fpdf2/Tutorial.html) 
<br>

Basisdingen:
[How to iterate over all or certain columns of a df](https://www.geeksforgeeks.org/python/loop-or-iterate-over-all-or-certain-columns-of-a-dataframe-in-python-pandas/) 
<br>
Om naar te kijken: <br>
-  [Google resultaten concepts](https://www.google.com/search?q=app+pc+for+brainstorming+with+drawing+tablet&num=10&sca_esv=9151e0e90600ee3c&sxsrf=ANbL-n7LjWhKl-hEHRYIRnvv6TYRhGIzTA%3A1774340841409&ei=6UrCaenXGMqLi-gPo-_UgAM&biw=1712&bih=1326&ved=0ahUKEwip8L_cjriTAxXKxQIHHaM3FTAQ4dUDCBE&uact=5&oq=app+pc+for+brainstorming+with+drawing+tablet&gs_lp=Egxnd3Mtd2l6LXNlcnAiLGFwcCBwYyBmb3IgYnJhaW5zdG9ybWluZyB3aXRoIGRyYXdpbmcgdGFibGV0MgUQIRigATIFECEYoAEyBRAhGKABSMY-UABY1j1wBngBkAEAmAFwoAGwHaoBBDQ5LjG4AQPIAQD4AQGYAjigArUfwgILEAAYgAQYkQIYigXCAgoQABiABBhDGIoFwgIQEC4YgAQY0QMYQxjHARiKBcICBRAAGIAEwgILEC4YgAQY0QMYxwHCAgUQLhiABMICBhAAGBYYHsICBxAAGIAEGA3CAgYQABgNGB7CAgsQABiABBiGAxiKBcICBRAAGO8FwgIIEAAYgAQYogTCAgcQIRigARgKwgIFECEYnwXCAgQQIRgVmAMAkgcENTQuMqAH3JsCsgcENDguMrgHkh_CBwkwLjI5LjI2LjHIB6EBgAgA&sclient=gws-wiz-serp)
<br>

Aantekeningen:
<br>

Hoe zorg ik dat ik meerdere inputs df's kan verwerken in één uiteindelijke score?


In [13]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fpdf import FPDF
import tempfile
from pathlib import Path
import os
import glob

In [14]:
#Tijdelijke oplossing totdat ik andere manier heb gevonden om input te krijgen
folder_path = r'c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\Testdingen'
pattern = os.path.join(folder_path, '*.csv')
csv_files = glob.glob(pattern)


print(csv_files)

['c:\\Users\\hans_\\Documents\\GitHub\\Stakeholder_analysis\\Testdingen\\input_stakeholders.csv', 'c:\\Users\\hans_\\Documents\\GitHub\\Stakeholder_analysis\\Testdingen\\input_stakeholders_2.csv', 'c:\\Users\\hans_\\Documents\\GitHub\\Stakeholder_analysis\\Testdingen\\input_stakeholders_3.csv']


In [15]:
#Create function to synthesize assesment input
def synthesize_assessment_input(file_path):
    pattern = os.path.join(file_path, '*.csv')
    csv_files = glob.glob(pattern)

    all_data = []

    if not csv_files:
        return FileNotFoundError(f"No CSV files found in the folder: {folder_path}") 
    #Doorloop alle csv bestanden en voeg ze samen in een dataframe
    for file in csv_files:
        temp_df = pd.read_csv(file, sep=';')
        all_data.append(temp_df)

    #Combineer alle dataframes in één dataframe
    combined_df = pd.concat(all_data, ignore_index=True)

    #Logica om de gecombineerde dataframe te verwerken en te synthetiseren
    aggregated_logic = {
        'formeel': 'mean',
        'informatie': 'mean',
        'informeel': 'mean',
        'legitimiteit': 'mean',
        'betrokkenheid': 'mean',
        'waarom': lambda x: ' | '.join(set(x))
    }

    synthesis = combined_df.groupby('stakeholder').agg(aggregated_logic).reset_index()
    #Mogelijk later toevoegen om af te ronden
    return synthesis

In [16]:
df = synthesize_assessment_input(r'c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\Testdingen')

In [17]:
#Check for Apple/WINDOWS path issues
print('cwd=', os.getcwd())
print(os.path.exists(r'c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\Testdingen\.ipynb_checkpoints\input_stakeholders-checkpoint.csv'))


cwd= C:\Users\hans_
True


In [18]:
def load_data(file_path):
    try:
        return pd.read_csv(file_path, sep=';')
    except FileNotFoundError:
        return pd.DataFrame({"StakeHolder": ['Project']}) #temp 



In [19]:
#Structureren van data:
def data_structure(df):
    columns = df.columns.tolist()
    for col in columns:
        try:
            all_counts = df[col].value_counts()
            return all_counts
        except ValueError:
            print('wrong values')

In [20]:
#Logica om strategie te bepalen op basis van power en interest score
def get_strategy(power, interest): #nodig: power en interest score op basis van input)
    if power >= 4 and interest >= 4: return "Manage closely"
    if power >= 4 and interest < 4: return "keep satisfied"
    if power < 4 and interest >= 4: return "keep informed"
    return "Monitor only"




In [21]:
def create_power_interest_columns(dataFrame):
    power_columns = ['formeel', 'informatie', 'informeel','legitimiteit']
    dataFrame['power_scores'] = dataFrame[power_columns].mean(axis=1)
    dataFrame['interest'] = dataFrame['betrokkenheid']

In [22]:
def create_matrix_plot(df):
    fig, ax = plt.subplots(figsize=(6,4))
    ax.scatter(df['power_scores'], df['interest'], c='blue')

    #Kwadranten indelen
    plt.axhline(3, color='black', linewidth=1)
    plt.axvline(3, color='black', linewidth=1)
    plt.xlim(1,5)
    plt.ylim(1,5)

    #Dit deel is voor de labels en titels, kan later nog mooier gemaakt worden
    plt.xlabel('power (1-5)')
    plt.ylabel('interest (1-5)')
    plt.title('Stakeholder map')

    for i, txt in enumerate(df['stakeholder']):
        ax.annotate(txt, (df['power_scores'].iat[i], df['interest'].iat[i])) #.iat werkt als iloc, maar dan voor specifieke cellen, niet hele rijen of kolommen

    plt.tight_layout()
    plt.show()
    plot_path = tempfile.NamedTemporaryFile(delete=False, suffix=".png").name
    plt.savefig(plot_path)
    print(f"File location: {plot_path}")
    return plot_path

In [23]:
create_power_interest_columns(df)
df['strategy'] = df.apply(lambda row: get_strategy(row['power_scores'], row['interest']), axis=1)



for i, row in df.iterrows():
    print(f"Stakeholder: {row['stakeholder']}, power: {row['power_scores']}, interest: {row['interest']}, Strategy: {row['strategy']}")


Stakeholder: FNV, power: 3.4166666666666665, interest: 3.3333333333333335, Strategy: Monitor only
Stakeholder: Stakeholder_d, power: 4.25, interest: 3.0, Strategy: keep satisfied
Stakeholder: VNG, power: 2.833333333333333, interest: 3.3333333333333335, Strategy: Monitor only
Stakeholder: VNO-NCW, power: 3.1666666666666665, interest: 3.0, Strategy: Monitor only


In [24]:
#Plot stakeholders op kaart
create_matrix_plot(df)


File location: C:\Users\hans_\AppData\Local\Temp\tmp9jvk1yf8.png


C:\Users\hans_\AppData\Local\Temp\ipykernel_9416\72303543.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'C:\\Users\\hans_\\AppData\\Local\\Temp\\tmp9jvk1yf8.png'

**Hoe cell-grootte kiezen?**
<br>
- Als regel: Kies een cell die 1,2 tot 1,5 keer zo groot is als de fontgrootte.
- De cellen zijn in mm
- 1 pt = 0,35 mm of 1 mm = 2.83 pt

<br>

**Pagina-grootte**
<br>
Afmetingen:
- Hoogte Height = 297
- Breedte = 210
- Top/link/recht marge = 10
- Bodemmarge = 20.002499999999998
- met pdf.get_y() en pdf.get_x() weet je waar je cursos in de pdf zit


In [ ]:
def create_test_pdf(data):
    #Start pdf
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font('Atmos', size=12)

    #Koptekst
    pdf.set_font('Atmos', 'B', size=12)
    pdf.cell(200, 10, txt= 'Stakeholder Analyse Rapport (test)', new_x='LMARGIN', new_y='NEXT', align='C')
    pdf.ln(10)

    #Tekst met stakeholders en strategieën
    pdf.set_font('Atmos', size=12)
    for i, row in data.iterrows():
        pdf.set_font('Atmos', 'B', size=12)
        pdf.cell(0, )



SyntaxError: incomplete input (1057317766.py, line 13)

In [36]:
pdf = FPDF()
print(pdf.h)
print(pdf.t_margin)
print(pdf.b_margin)
print(pdf.w)
print(pdf.l_margin)


297.0000833333333
10.001249999999999
20.002499999999998
210.0015555555555
10.001249999999999


In [ ]:
#PDF genereren in streamlit
st.title("Rapport Stakeholder Analyse")
st.write('Overzicht met de belangrijkste stakeholders en hun strategieën.')
with st.form('analysis_form'):
    for index, row in df.iterrows():
        st.write(f"Stakeholder: {row['stakeholder']}, power: {row['power_scores']}, interest: {row['interest']}, Strategy: {row['strategy']}")
        
